## 7.2 — Trusted Data Context

### Objective
Prepare accurate, relevant, and minimal business data for GenAI.

### Data source
MySQL database: customer_360

### Data access strategy
- Use existing analytical views as the primary source.
- Query only the fields required for each AI task.
- Use base tables only when a required detail is missing from the views.
- Keep raw datasets out of trusted AI context.
- Validate numerical outputs against SQL results before generating explanations.
- Do not send unnecessary personal information to an external LLM.

In [ ]:
import mysql.connector
from getpass import getpass

username = input("MySQL username: ")
password = getpass("MySQL password: ") 

DB_CONFIG = {
"host": "localhost",
"port": 3306,
"user": username,
"password": password,
"database": "customer_360"
}


In [ ]:
ALLOWED_VIEWS = {
"vw_customer_360",
"vw_customer_360_segmented",
"vw_customer_engagement_metrics",
"vw_customer_order_metrics",
"vw_customer_support_metrics",
"vw_monthly_revenue",
"vw_product_performance"
}

def get_connection():
    return mysql.connector.connect(**DB_CONFIG)

# Test the connection
connection = get_connection()
print("MySQL connection successful:", connection.is_connected())
connection.close()

MySQL connection successful: True


In [ ]:
CUSTOMER_VIEWS = [
"vw_customer_360",
"vw_customer_360_segmented",
"vw_customer_engagement_metrics",
"vw_customer_order_metrics",
"vw_customer_support_metrics"
]

# Exclude personal identifiers from context prepared for GenAI.
PRIVATE_FIELDS = {
    "email", "phone", "phone_number", "address",
    "first_name", "last_name", "full_name",
    "customer_name", "device_id", "device_ids"
}


def get_safe_columns(cursor, view_name):
    if view_name not in ALLOWED_VIEWS:
        raise ValueError("View is not in the approved list.")

    cursor.execute(f"SHOW COLUMNS FROM `{view_name}`")
    columns = [row[0] for row in cursor.fetchall()]

    return [
        col for col in columns
        if col.lower() not in PRIVATE_FIELDS
    ]


def fetch_customer_context(customer_id):
    """Retrieve one customer's relevant metrics from approved views."""
    context = {}
    connection = get_connection()

    try:
        for view_name in CUSTOMER_VIEWS:
            cursor = connection.cursor()
            columns = get_safe_columns(cursor, view_name)
            cursor.close()

            if "customer_id" not in columns:
                continue

            selected_columns = ", ".join(
                f"`{col}`" for col in columns
            )

            cursor = connection.cursor(dictionary=True)
            query = (
                f"SELECT {selected_columns} "
                f"FROM `{view_name}` "
                "WHERE `customer_id` = %s LIMIT 1"
            )
            cursor.execute(query, (customer_id,))
            context[view_name] = cursor.fetchone()
            cursor.close()

        return context

    finally:
        connection.close()


def fetch_business_context(view_name, limit=10):
    """Retrieve a small, approved sample for business analysis."""
    if view_name not in ALLOWED_VIEWS:
        raise ValueError("View is not in the approved list.")

    if not isinstance(limit, int) or not 1 <= limit <= 100:
        raise ValueError("Limit must be an integer from 1 to 100.")

    connection = get_connection()

    try:
        cursor = connection.cursor()
        columns = get_safe_columns(cursor, view_name)
        cursor.close()

        if not columns:
            return []

        selected_columns = ", ".join(
            f"`{col}`" for col in columns
        )

        cursor = connection.cursor(dictionary=True)
        cursor.execute(
            f"SELECT {selected_columns} "
            f"FROM `{view_name}` LIMIT {limit}"
        )
        results = cursor.fetchall()
        cursor.close()

        return results

    finally:
        connection.close()

In [4]:
monthly_sample = fetch_business_context(
    "vw_monthly_revenue", limit=5
)

product_sample = fetch_business_context(
    "vw_product_performance", limit=5
)

print("Monthly revenue sample:")
print(monthly_sample)

print("\nProduct performance sample:")
print(product_sample)

Monthly revenue sample:
[{'ym': '2023-12', 'order_year': 2023, 'order_month': 12, 'total_revenue': Decimal('365613492.27'), 'total_orders': 7256}, {'ym': '2024-01', 'order_year': 2024, 'order_month': 1, 'total_revenue': Decimal('411999016.77'), 'total_orders': 7889}, {'ym': '2024-02', 'order_year': 2024, 'order_month': 2, 'total_revenue': Decimal('341366277.06'), 'total_orders': 7272}, {'ym': '2024-03', 'order_year': 2024, 'order_month': 3, 'total_revenue': Decimal('391021050.25'), 'total_orders': 7890}, {'ym': '2024-04', 'order_year': 2024, 'order_month': 4, 'total_revenue': Decimal('360268756.86'), 'total_orders': 7445}]

Product performance sample:
[{'product_id': 'PROD-0001', 'product_name': 'open beaautiful', 'category': 'clothing', 'price': Decimal('32309.95'), 'total_orders': 567, 'total_quantity': Decimal('1339.00'), 'total_revenue': Decimal('27928620.97')}, {'product_id': 'PROD-0002', 'product_name': 'race third', 'category': 'automotive', 'price': Decimal('47346.82'), 'total_

In [5]:
connection = get_connection()

try:
    cursor = connection.cursor()
    cursor.execute("""
        SELECT customer_id
        FROM vw_customer_360
        WHERE customer_id IS NOT NULL
        LIMIT 1
    """)
    row = cursor.fetchone()
    cursor.close()
finally:
    connection.close()

if row is None:
    print("No customer ID found in vw_customer_360.")
else:
    sample_customer_id = row[0]
    customer_context = fetch_customer_context(sample_customer_id)

    print("Customer context retrieval successful.")
    for view_name, data in customer_context.items():
        print(f"\n{view_name}:")
        print(data)

Customer context retrieval successful.

vw_customer_360:
{'customer_id': '0000900d-5512-4cdf-8ea5-eded823b2ff8', 'gender': 'f', 'city': 'east peterland', 'state': 'massachusetts', 'country': 'french guiana', 'source': 'referral', 'first_order_date': datetime.date(2024, 4, 5), 'last_order_date': datetime.date(2024, 12, 14), 'customer_lifetime_days': 253, 'total_orders': 3, 'total_revenue': Decimal('117943.26'), 'total_quantity': Decimal('3.00'), 'avg_order_value': Decimal('39314.42'), 'days_since_last_order': 352, 'engagement_events': 11, 'support_tickets': 0, 'avg_resolution_time': None, 'positive_tickets': Decimal('0'), 'negative_tickets': Decimal('0'), 'neutral_tickets': Decimal('0')}

vw_customer_360_segmented:
{'customer_id': '0000900d-5512-4cdf-8ea5-eded823b2ff8', 'gender': 'f', 'city': 'east peterland', 'state': 'massachusetts', 'country': 'french guiana', 'source': 'referral', 'first_order_date': datetime.date(2024, 4, 5), 'last_order_date': datetime.date(2024, 12, 14), 'custome

In [6]:
#%pip install -U google-genai python-dotenv

In [7]:
import os
print (os.getcwd())

d:\Customer_360\07_GenAI


In [8]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", bool(api_key))

API key loaded: True


In [9]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply with exactly: GenAI connection successful."
)

print(response.text)

GenAI connection successful.


In [28]:
import json
from decimal import Decimal
from datetime import date, datetime


def to_json_safe(value):
    """Convert database values into JSON-compatible values."""
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, (date, datetime)):
        return value.isoformat()
    return value


def generate_customer_360_summary(customer_id):
    # Retrieve metrics from our existing MySQL views.
    context = fetch_customer_context(customer_id)

    customer = context.get("vw_customer_360")
    segments = context.get("vw_customer_360_segmented") or {}

    if not customer:
        return "No customer record found in vw_customer_360."

    # Allowlist only the business metrics needed by the AI.
    fields = [
        "first_order_date",
        "last_order_date",
        "customer_lifetime_days",
        "total_orders",
        "total_revenue",
        "total_quantity",
        "avg_order_value",
        "days_since_last_order",
        "engagement_events",
        "support_tickets",
        "avg_resolution_time",
        "positive_tickets",
        "negative_tickets",
        "neutral_tickets",
    ]

    metrics = {
        field: to_json_safe(customer.get(field))
        for field in fields
    }

    metrics["segments"] = {
        field: segments.get(field)
        for field in [
            "purchase_frequency_segment",
            "engagement_segment",
            "support_segment",
        ]
    }

    context_json = json.dumps(metrics, indent=2)

    prompt = f"""
You are a careful Customer 360 business analyst.

Create a concise summary using ONLY the supplied data.

Rules:
- Do not invent figures, explanations, or customer characteristics.
- Do not claim that correlation proves causation.
- Treat null values as unavailable, not automatically as zero.
- Interpret days_since_last_order as the value reported by the dataset.
- Give recommendations as suggestions, not established facts.
- Do not reproduce database identifiers.

Use these sections:
1. Customer Snapshot
2. Purchase Behaviour
3. Engagement
4. Support Experience
5. Suggested Next Action
6. Data Limitations

Keep the language clear and professional.

Customer metrics:
{context_json}
"""

    response = generate_with_retry(prompt)
    return response.text


# Test with the sample customer retrieved earlier.
summary = generate_customer_360_summary(sample_customer_id)
print(summary)

### 1. Customer Snapshot
* **Customer Lifetime Days:** 253 days (First order: 2024-04-05; Last order: 2024-12-14)
* **Segments:** 
  * Purchase Frequency: 3-5 Orders
  * Engagement: High
  * Support: No Tickets

### 2. Purchase Behaviour
* **Total Orders:** 3
* **Total Revenue:** $117,943.26
* **Total Quantity:** 3.0
* **Average Order Value:** $39,314.42
* **Days Since Last Order:** 352 days

### 3. Engagement
* **Engagement Events:** 11

### 4. Support Experience
* **Support Tickets:** 0
* **Positive, Negative, and Neutral Tickets:** 0.0 each

### 5. Suggested Next Action
* Suggest reviewing the account to understand the elapsed time since the last order (352 days) despite high engagement events.

### 6. Data Limitations
* Average resolution time is unavailable due to a null value.


## 7.5 — Customer Segmentation Insights

### Objective
Use GenAI to explain the business characteristics and value of customer segments.

### Data source
MySQL view: vw_customer_360_segmented

### Approach
1. Aggregate customer metrics by purchase frequency, engagement, and support segment.
2. Send only aggregated results to the LLM.
3. Generate evidence-based observations and suggested business actions.
4. Validate every numerical claim against the SQL results.

In [11]:
from decimal import Decimal


def fetch_segment_summaries():
    segment_columns = {
        "purchase_frequency": "purchase_frequency_segment",
        "engagement": "engagement_segment",
        "support": "support_segment",
    }

    summaries = {}
    connection = get_connection()

    try:
        for name, column in segment_columns.items():
            query = f"""
                SELECT
                    `{column}` AS segment,
                    COUNT(DISTINCT customer_id) AS customer_count,
                    SUM(COALESCE(total_revenue, 0)) AS total_revenue,
                    AVG(COALESCE(total_orders, 0)) AS avg_orders,
                    AVG(COALESCE(engagement_events, 0))
                        AS avg_engagement_events,
                    AVG(COALESCE(support_tickets, 0))
                        AS avg_support_tickets
                FROM vw_customer_360_segmented
                GROUP BY `{column}`
                ORDER BY customer_count DESC
            """

            cursor = connection.cursor(dictionary=True)
            cursor.execute(query)
            rows = cursor.fetchall()
            cursor.close()

            # Convert database Decimal values for JSON/API use.
            for row in rows:
                for key, value in row.items():
                    if isinstance(value, Decimal):
                        row[key] = float(value)

            summaries[name] = rows

        return summaries

    finally:
        connection.close()


segment_summaries = fetch_segment_summaries()

for dimension, rows in segment_summaries.items():
    print(f"\n{dimension.upper()} SEGMENTS")
    for row in rows:
        print(row)


PURCHASE_FREQUENCY SEGMENTS
{'segment': '3-5 Orders', 'customer_count': 21856, 'total_revenue': 4458381417.9, 'avg_orders': 4.0988, 'avg_engagement_events': 7.0165, 'avg_support_tickets': 0.593}
{'segment': '6-10 Orders', 'customer_count': 19748, 'total_revenue': 7119735321.66, 'avg_orders': 7.2248, 'avg_engagement_events': 7.2928, 'avg_support_tickets': 0.6348}
{'segment': '1-2 Orders', 'customer_count': 4643, 'total_revenue': 407575418.98, 'avg_orders': 1.7219, 'avg_engagement_events': 6.935, 'avg_support_tickets': 0.6028}
{'segment': '11+ Orders', 'customer_count': 1696, 'total_revenue': 1056957438.27, 'avg_orders': 12.4511, 'avg_engagement_events': 10.671, 'avg_support_tickets': 0.9169}
{'segment': 'No Orders', 'customer_count': 257, 'total_revenue': 0.0, 'avg_orders': 0.0, 'avg_engagement_events': 6.9183, 'avg_support_tickets': 0.5798}

ENGAGEMENT SEGMENTS
{'segment': 'Medium', 'customer_count': 28286, 'total_revenue': 7392315649.67, 'avg_orders': 5.2735, 'avg_engagement_events':

In [12]:
import json

segment_data = json.dumps(segment_summaries, indent=2)

prompt = f"""
You are a careful Customer 360 business analyst.

Analyze the following aggregated customer-segment metrics.

Tasks:
1. Explain the main characteristics of each segment.
2. Identify which segments contribute most to total revenue.
3. Compare average orders and engagement across segments.
4. Discuss support-ticket patterns where the evidence permits.
5. Suggest practical business actions for each major finding.

Rules:
- Use only the supplied data.
- Do not invent percentages, benchmarks, or causal explanations.
- Distinguish total revenue from average revenue per customer.
- Do not claim that high engagement causes higher revenue.
- Mention limitations where relevant.
- Keep findings concise and suitable for a business report.
- Do not assume or invent a currency symbol or currency name.
- Use neutral revenue values unless the data source defines the currency.
- Avoid words such as "closely" unless the numerical difference supports them.
- Do not describe customers as high-value without comparing
  an appropriate value metric, such as average revenue per customer.
- Clearly distinguish measured facts from suggested business actions.

Aggregated SQL results:
{segment_data}
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt,
)

print(response.text)

# Customer 360 Business Analysis Report

---

## 1. Main Characteristics of Each Segment

### Purchase Frequency Segments
* **No Orders (257 customers):** Account for 0.0 total revenue, 0.0 average orders, 6.9183 average engagement events, and 0.5798 average support tickets.
* **1-2 Orders (4,643 customers):** Account for 407,575,418.98 total revenue, 1.7219 average orders, 6.935 average engagement events, and 0.6028 average support tickets.
* **3-5 Orders (21,856 customers):** Account for 4,458,381,417.9 total revenue, 4.0988 average orders, 7.0165 average engagement events, and 0.593 average support tickets.
* **6-10 Orders (19,748 customers):** Account for 7,119,735,321.66 total revenue, 7.2248 average orders, 7.2928 average engagement events, and 0.6348 average support tickets.
* **11+ Orders (1,696 customers):** Account for 1,056,957,438.27 total revenue, 12.4511 average orders, 10.671 average engagement events, and 0.9169 average support tickets.

### Engagement Segments
* **Low 

## 7.6 — Revenue and Business Insights

### Objective
Generate evidence-based business insights from monthly revenue
and product performance data.

### Data sources
- vw_monthly_revenue
- vw_product_performance

### Approach
1. Retrieve monthly revenue history from MySQL.
2. Aggregate product performance by category.
3. Identify the top 10 products by revenue.
4. Validate revenue totals across the two views.
5. Use GenAI to explain trends and suggest business actions.
6. Verify the AI's claims against the SQL results.

In [13]:
from decimal import Decimal
import json


def prepare_revenue_business_context():
    connection = get_connection()

    try:
        cursor = connection.cursor(dictionary=True)

        # 1. Monthly revenue history
        cursor.execute("""
            SELECT
                ym,
                order_year,
                order_month,
                total_revenue,
                total_orders
            FROM vw_monthly_revenue
            ORDER BY order_year, order_month
        """)
        monthly_rows = cursor.fetchall()

        # 2. Product and category performance
        cursor.execute("""
            SELECT
                category,
                COUNT(DISTINCT product_id) AS product_count,
                SUM(total_revenue) AS total_revenue,
                SUM(total_quantity) AS total_quantity,
                SUM(total_orders) AS total_orders
            FROM vw_product_performance
            GROUP BY category
            ORDER BY total_revenue DESC
        """)
        category_rows = cursor.fetchall()

        # 3. Top 10 products by revenue
        cursor.execute("""
            SELECT
                product_id,
                product_name,
                category,
                total_revenue,
                total_quantity,
                total_orders
            FROM vw_product_performance
            ORDER BY total_revenue DESC
            LIMIT 10
        """)
        top_products = cursor.fetchall()

        cursor.close()

        # Convert Decimal values into JSON-compatible numbers.
        def convert_rows(rows):
            converted = []
            for row in rows:
                converted.append({
                    key: float(value)
                    if isinstance(value, Decimal)
                    else value
                    for key, value in row.items()
                })
            return converted

        monthly_rows = convert_rows(monthly_rows)
        category_rows = convert_rows(category_rows)
        top_products = convert_rows(top_products)

        monthly_total = sum(
            row["total_revenue"] for row in monthly_rows
        )
        category_total = sum(
            row["total_revenue"] for row in category_rows
        )

        return {
            "monthly_revenue": monthly_rows,
            "category_performance": category_rows,
            "top_10_products": top_products,
            "validation": {
                "monthly_revenue_total": round(monthly_total, 2),
                "product_category_revenue_total": round(
                    category_total, 2
                ),
                "revenue_totals_match": abs(
                    monthly_total - category_total
                ) < 0.01,
                "monthly_periods": len(monthly_rows),
                "categories": len(category_rows),
                "top_products_returned": len(top_products)
            }
        }

    finally:
        connection.close()


revenue_business_context = prepare_revenue_business_context()

print(json.dumps(
    revenue_business_context["validation"],
    indent=2
))

print("\nCATEGORY PERFORMANCE")
print(json.dumps(
    revenue_business_context["category_performance"],
    indent=2
))

print("\nTOP 10 PRODUCTS")
print(json.dumps(
    revenue_business_context["top_10_products"],
    indent=2
))

{
  "monthly_revenue_total": 9100659243.71,
  "product_category_revenue_total": 13042649596.81,
  "revenue_totals_match": false,
  "monthly_periods": 25,
  "categories": 9,
  "top_products_returned": 10
}

CATEGORY PERFORMANCE
[
  {
    "category": "kitchen",
    "product_count": 68,
    "total_revenue": 1795990060.39,
    "total_quantity": 82978.0,
    "total_orders": 35573.0
  },
  {
    "category": "clothing",
    "product_count": 69,
    "total_revenue": 1786580099.8,
    "total_quantity": 84928.0,
    "total_orders": 36167.0
  },
  {
    "category": "beauty",
    "product_count": 60,
    "total_revenue": 1562736230.44,
    "total_quantity": 74824.0,
    "total_orders": 31563.0
  },
  {
    "category": "home",
    "product_count": 59,
    "total_revenue": 1528823145.29,
    "total_quantity": 71740.0,
    "total_orders": 30458.0
  },
  {
    "category": "electronics",
    "product_count": 57,
    "total_revenue": 1509712538.93,
    "total_quantity": 70252.0,
    "total_orders": 3000

In [14]:
from decimal import Decimal

connection = get_connection()

try:
    cursor = connection.cursor(dictionary=True)

    cursor.execute("""
        SELECT
            COUNT(*) AS total_order_rows,
            COUNT(order_date) AS orders_with_date,
            SUM(order_date IS NULL) AS orders_without_date,
            SUM(order_amount) AS all_orders_revenue,
            SUM(
                CASE
                    WHEN order_date IS NOT NULL
                    THEN order_amount
                    ELSE 0
                END
            ) AS revenue_with_date,
            SUM(
                CASE
                    WHEN order_date IS NULL
                    THEN order_amount
                    ELSE 0
                END
            ) AS revenue_without_date
        FROM orders
    """)

    validation = cursor.fetchone()
    cursor.close()

    for key, value in validation.items():
        if isinstance(value, Decimal):
            validation[key] = float(value)

    print("ORDER REVENUE VALIDATION")
    for key, value in validation.items():
        print(f"{key}: {value}")

finally:
    connection.close()

ORDER REVENUE VALIDATION
total_order_rows: 300000
orders_with_date: 210000
orders_without_date: 90000.0
all_orders_revenue: 13042649596.81
revenue_with_date: 9100659243.71
revenue_without_date: 3941990353.1


In [15]:
import json
from decimal import Decimal

connection = get_connection()

try:
    cursor = connection.cursor(dictionary=True)
    cursor.execute("""
        SELECT
            COUNT(*) AS total_order_rows,
            SUM(order_date IS NOT NULL) AS orders_with_date,
            SUM(order_date IS NULL) AS orders_without_date,
            SUM(order_amount) AS all_orders_revenue,
            SUM(
                CASE WHEN order_date IS NOT NULL
                     THEN order_amount ELSE 0 END
            ) AS revenue_with_date,
            SUM(
                CASE WHEN order_date IS NULL
                     THEN order_amount ELSE 0 END
            ) AS revenue_without_date
        FROM orders
    """)
    date_reconciliation = cursor.fetchone()
    cursor.close()
finally:
    connection.close()

for key, value in date_reconciliation.items():
    if isinstance(value, Decimal):
        date_reconciliation[key] = float(value)

revenue_business_context["date_reconciliation"] = {
    key: round(value, 2) if isinstance(value, float) and
    "revenue" in key else value
    for key, value in date_reconciliation.items()
}

revenue_data = json.dumps(revenue_business_context, indent=2)

prompt = f"""
You are a careful business analyst working on a Customer 360 project.

Analyze the provided monthly revenue, product, category, and
data-reconciliation results.

Produce:
1. Overall Revenue Performance
2. Monthly Revenue Trends
3. Category Performance
4. Top 10 Product Insights
5. Practical Business Recommendations
6. Data Limitations

Rules:
- Use only the supplied data.
- Do not invent currency symbols, benchmarks, or figures.
- Monthly revenue covers only orders with non-null order dates.
- 90,000 orders have no date and contribute 3,941,990,353.10
  in revenue. The monthly total excludes this revenue.
- Do not describe this difference as unexplained or as lost revenue.
- The product-category totals include products with a null category.
- Do not treat an incomplete latest month as a confirmed decline.
- Revenue does not establish profitability.
- Separate measured findings from suggested actions.
- Mention that missing dates limit the completeness of trend analysis.

Business data:
{revenue_data}
"""

try:
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )
    print(response.text)
except Exception as e:
    print("AI report generation failed:")
    print(str(e))

Based on the Customer 360 project data provided, here is the detailed business analysis:

### 1. Overall Revenue Performance
* **Total Revenue Scope:** The comprehensive revenue across all 300,000 order rows totals 13,042,649,596.81.
* **Dated Order Revenue:** Orders with valid order dates account for 210,000 orders and generate 9,100,659,243.71 in revenue (spanning 25 monthly periods from December 2023 through December 2025).
* **Undated Order Revenue:** There are 90,000 orders without a date, contributing 3,941,990,353.10 in revenue. 
* **Profitability:** This analysis measures revenue only; revenue does not establish profitability.

### 2. Monthly Revenue Trends
* **Observation Period:** Monthly figures cover 25 periods (December 2023 to December 2025) using only orders with non-null order dates.
* **Monthly Range:** Monthly revenue generally fluctuates between approximately 341.3 million and 412.0 million, with monthly order counts typically ranging between 7,100 and 7,900. 
* **La

### Step 7.6 — Validation Correction: Top 10 Products

For product `PROD-0313` (“professor reaach”), the correct values from the MySQL `vw_product_performance` view are:

- **Revenue:** 33,318,253.91
- **Quantity sold:** 1,220
- **Orders:** 526

The original AI report incorrectly labelled the revenue value as quantity. The values above have been checked against the SQL results.

## 7.7 — Support Intelligence

### Objective
Generate evidence-based insights from customer support data.

### Data sources
- support_tickets
- vw_customer_360_segmented

### Analysis
1. Overall support volume and resolution time.
2. Ticket distribution by issue type and sentiment.
3. Resolution performance across issue types.
4. Support-ticket patterns by customer segment.
5. Practical suggestions for improving support operations.

### Validation
Verify numerical claims against SQL results and flag
missing resolution data and small customer segments.

In [16]:
from decimal import Decimal
from datetime import date, datetime
import json


def prepare_support_context():
    connection = get_connection()

    try:
        cursor = connection.cursor(dictionary=True)

        # 1. Overall support metrics
        cursor.execute("""
            SELECT
                COUNT(DISTINCT ticket_id) AS total_tickets,
                COUNT(DISTINCT customer_id) AS customers_with_tickets,
                AVG(resolution_time_hours) AS avg_resolution_hours,
                SUM(resolution_time_hours IS NULL)
                    AS missing_resolution_times,
                COUNT(ticket_resolved) AS tickets_with_resolved_date
            FROM support_tickets
        """)
        overview = cursor.fetchone()

        # 2. Issue-type patterns
        cursor.execute("""
            SELECT
                issue_type,
                COUNT(DISTINCT ticket_id) AS ticket_count,
                AVG(resolution_time_hours) AS avg_resolution_hours,
                SUM(resolution_time_hours IS NULL)
                    AS missing_resolution_times
            FROM support_tickets
            GROUP BY issue_type
            ORDER BY ticket_count DESC
        """)
        issue_summary = cursor.fetchall()

        # 3. Sentiment patterns
        cursor.execute("""
            SELECT
                sentiment,
                COUNT(DISTINCT ticket_id) AS ticket_count,
                AVG(resolution_time_hours) AS avg_resolution_hours
            FROM support_tickets
            GROUP BY sentiment
            ORDER BY ticket_count DESC
        """)
        sentiment_summary = cursor.fetchall()

        # 4. Customer support segments
        cursor.execute("""
            SELECT
                support_segment,
                COUNT(DISTINCT customer_id) AS customer_count,
                SUM(COALESCE(total_revenue, 0)) AS total_revenue,
                AVG(COALESCE(total_orders, 0)) AS avg_orders,
                AVG(COALESCE(engagement_events, 0))
                    AS avg_engagement_events,
                AVG(COALESCE(support_tickets, 0))
                    AS avg_support_tickets
            FROM vw_customer_360_segmented
            GROUP BY support_segment
            ORDER BY customer_count DESC
        """)
        segment_summary = cursor.fetchall()

        cursor.close()

        def convert(value):
            if isinstance(value, Decimal):
                return float(value)
            if isinstance(value, (date, datetime)):
                return value.isoformat()
            return value

        def convert_rows(rows):
            return [
                {key: convert(value) for key, value in row.items()}
                for row in rows
            ]

        return {
            "overview": {
                key: convert(value)
                for key, value in overview.items()
            },
            "issue_types": convert_rows(issue_summary),
            "sentiment": convert_rows(sentiment_summary),
            "customer_support_segments": convert_rows(segment_summary)
        }

    finally:
        connection.close()


support_context = prepare_support_context()

print(json.dumps(support_context, indent=2))

{
  "overview": {
    "total_tickets": 30000,
    "customers_with_tickets": 22149,
    "avg_resolution_hours": 121.008733,
    "missing_resolution_times": 0.0,
    "tickets_with_resolved_date": 21000
  },
  "issue_types": [
    {
      "issue_type": "refund",
      "ticket_count": 7572,
      "avg_resolution_hours": 121.428024,
      "missing_resolution_times": 0.0
    },
    {
      "issue_type": "product",
      "ticket_count": 7542,
      "avg_resolution_hours": 121.210554,
      "missing_resolution_times": 0.0
    },
    {
      "issue_type": "payment",
      "ticket_count": 7464,
      "avg_resolution_hours": 120.539657,
      "missing_resolution_times": 0.0
    },
    {
      "issue_type": "delay",
      "ticket_count": 7422,
      "avg_resolution_hours": 120.847615,
      "missing_resolution_times": 0.0
    }
  ],
  "sentiment": [
    {
      "sentiment": "negative",
      "ticket_count": 10019,
      "avg_resolution_hours": 120.823136
    },
    {
      "sentiment": "positive",

In [17]:
import json

support_data = json.dumps(support_context, indent=2)

prompt = f"""
You are a careful Customer 360 support analyst.

Analyze the supplied SQL results and produce a concise report with:

1. Overall Support Overview
2. Common Issue Types
3. Customer Sentiment
4. Resolution Efficiency
5. Support Patterns Across Customer Segments
6. Suggested Business Actions
7. Data Limitations

Rules:
- Use only the supplied metrics.
- Do not invent figures, currency symbols, or benchmarks.
- Distinguish ticket counts from unique customer counts.
- The data reports 30,000 tickets but only 21,000 non-null resolved dates.
- Resolution-time values are populated for all tickets despite the
  missing resolved dates. Flag this as a data-quality point and do
  not assume that every ticket is resolved.
- Treat unavailable metrics as unavailable, not as zero.
- Do not claim that ticket volume proves dissatisfaction or causes churn.
- Explicitly flag the 6+ Tickets segment because it contains only 7 customers.
- Distinguish observations from recommendations.
- Keep the report professional and suitable for a portfolio project
  and a data analyst interview.

Support data:
{support_data}
"""

try:
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )
    print(response.text)

except Exception as e:
    print("Support insight generation failed:")
    print(str(e))

# Customer 360 Support Analysis Report

## 1. Overall Support Overview
- **Ticket Volume:** A total of 30,000 support tickets were recorded across 22,149 unique customers who submitted at least one ticket.
- **Resolution Timelines:** The dataset reports an average resolution time of approximately 121.01 hours across all tickets. 
- **Data Integrity Observation:** Resolution-time values are populated for all 30,000 tickets, yet only 21,000 tickets contain a non-null resolved date. Consequently, resolution-time metrics should not be interpreted as applying exclusively to fully resolved tickets, as unresolved or active tickets appear to carry resolution-time figures.

## 2. Common Issue Types
Support volume is distributed almost evenly across four primary issue types:
- **Refund:** 7,572 tickets (average resolution time: 121.43 hours)
- **Product:** 7,542 tickets (average resolution time: 121.21 hours)
- **Payment:** 7,464 tickets (average resolution time: 120.54 hours)
- **Delay:** 7,422

## 7.8 — Data-Grounded Business Q&A

### Objective
Answer business questions using trusted MySQL results.

### Approach
1. Identify the business topic in the question.
2. Retrieve relevant metrics from approved SQL views.
3. Send only the relevant context to the LLM.
4. Generate a concise, evidence-based answer.
5. State limitations when the database cannot answer a question.

### Safety
- Use predefined SQL queries only.
- Do not execute unrestricted, model-generated SQL.
- Do not expose personal customer information.
- Do not invent missing figures or business facts.

In [18]:
def retrieve_qa_context(question):
    """Retrieve approved business context based on the question."""

    question = question.lower().strip()

    support_keywords = [
        "support", "ticket", "sentiment", "resolution",
        "refund", "payment issue", "customer service"
    ]

    segment_keywords = [
        "segment", "engagement", "purchase frequency",
        "repeat customer", "no orders"
    ]

    product_keywords = [
        "product", "category", "categories", "quantity",
        "best selling", "top selling", "top 10"
    ]

    revenue_keywords = [
        "revenue", "sales", "monthly", "month",
        "trend", "year", "growth"
    ]

    if any(word in question for word in support_keywords):
        return {
            "topic": "support",
            "source": "support_tickets and vw_customer_360_segmented",
            "data": prepare_support_context()
        }

    if any(word in question for word in segment_keywords):
        return {
            "topic": "customer_segments",
            "source": "vw_customer_360_segmented",
            "data": fetch_segment_summaries()
        }

    if any(word in question for word in product_keywords):
        business_data = prepare_revenue_business_context()

        return {
            "topic": "product_performance",
            "source": "vw_product_performance",
            "data": {
                "category_performance":
                    business_data["category_performance"],
                "top_10_products":
                    business_data["top_10_products"],
                "validation":
                    business_data["validation"]
            }
        }

    if any(word in question for word in revenue_keywords):
        business_data = prepare_revenue_business_context()

        return {
        "topic": "revenue",
        "source": (
            "vw_monthly_revenue, "
            "vw_product_performance, and orders"
        ),
        "data": {
            "monthly_revenue": business_data["monthly_revenue"],
            "validation": business_data["validation"],
            "date_reconciliation": fetch_date_reconciliation()
            }
        }

    return {
        "topic": "general_business",
        "source": "approved Customer 360 analytical views",
        "data": {
            "customer_segments": fetch_segment_summaries(),
            "support": prepare_support_context(),
            "revenue": {
                "validation":
                    prepare_revenue_business_context()["validation"]
            }
        }
    }

In [29]:
import json


def answer_business_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Please enter a business question.")

    context = retrieve_qa_context(question)

    prompt = f"""
You are a Customer 360 business analyst answering a user's question.

Question:
{question}

Trusted database context:
{json.dumps(context, indent=2, default=str)}

Instructions:
- Answer the user's question directly.
- Use only the supplied database context.
- You may perform simple arithmetic using the supplied figures.
- Never invent figures, currency symbols, benchmarks, or explanations.
- Distinguish observations from recommendations.
- Mention limitations when relevant.
- If the context doesn't contain enough information, say so.
- Never claim that correlation establishes causation.
- Do not reveal database credentials or personal customer information.
- Keep the answer concise, clear, and interview-ready.
- For monthly revenue questions, use date_reconciliation if provided.
- 90,000 orders lack dates and account for 3,941,990,353.10
  in revenue; monthly revenue excludes this undated revenue.
- Do not describe this known difference as unexplained.
- Do not discuss top-product coverage unless the question asks
  about products or product-level analysis.
- Do not invent a currency symbol or currency name.
"""

    response = generate_with_retry(prompt)

    answer = response.text

# The dataset does not define a currency.
    for symbol in ["$", "₹", "€", "£"]:
        answer = answer.replace(symbol, "")

    return answer


print(answer_business_question(
    "Which product category generates the most revenue?"
))

The product category that generates the most revenue is **kitchen**, with a total revenue of **1,795,990,060.39** (generated across 68 products, 82,978 total quantity, and 35,573 total orders).


In [22]:
from decimal import Decimal


def fetch_date_reconciliation():
    connection = get_connection()

    try:
        cursor = connection.cursor(dictionary=True)

        cursor.execute("""
            SELECT
                COUNT(*) AS total_order_rows,
                COUNT(order_date) AS orders_with_date,
                SUM(order_date IS NULL) AS orders_without_date,
                SUM(order_amount) AS all_orders_revenue,
                SUM(
                    CASE WHEN order_date IS NOT NULL
                         THEN order_amount ELSE 0 END
                ) AS revenue_with_date,
                SUM(
                    CASE WHEN order_date IS NULL
                         THEN order_amount ELSE 0 END
                ) AS revenue_without_date
            FROM orders
        """)

        result = cursor.fetchone()
        cursor.close()

        return {
            key: round(float(value), 2)
            if isinstance(value, Decimal) else value
            for key, value in result.items()
        }

    finally:
        connection.close()

In [ ]:
questions = [
    "Which customer segment contributes the most revenue?",
    "Which support issue type takes the longest to resolve?",
    "What limitations should I consider when analysing monthly revenue?"
]

for question in questions:
    print(f"\nQUESTION: {question}")
    print(answer_business_question(question))
    print("-" * 70)

In [24]:
from decimal import Decimal

connection = get_connection()

try:
    cursor = connection.cursor(dictionary=True)
    cursor.execute("""
        SELECT ym, total_revenue, total_orders
        FROM vw_monthly_revenue
        WHERE ym = '2025-12'
    """)
    result = cursor.fetchone()
    cursor.close()

    if result:
        print("Source-of-truth result:")
        print("Month:", result["ym"])
        print("Revenue:", result["total_revenue"])
        print("Orders:", result["total_orders"])
    else:
        print("December 2025 was not found in the view.")

finally:
    connection.close()

Source-of-truth result:
Month: 2025-12
Revenue: 12794588.15
Orders: 268


In [25]:
from decimal import Decimal

connection = get_connection()

try:
    cursor = connection.cursor(dictionary=True)

    cursor.execute("""
        SELECT
            SUM(order_amount) AS all_revenue,
            SUM(CASE WHEN order_date IS NOT NULL
                     THEN order_amount ELSE 0 END) AS dated_revenue,
            SUM(CASE WHEN order_date IS NULL
                     THEN order_amount ELSE 0 END) AS undated_revenue
        FROM orders
    """)
    order_totals = cursor.fetchone()

    cursor.execute("""
        SELECT SUM(total_revenue) AS product_revenue
        FROM vw_product_performance
    """)
    product_total = cursor.fetchone()["product_revenue"]

    cursor.execute("""
        SELECT SUM(total_revenue) AS monthly_revenue
        FROM vw_monthly_revenue
    """)
    monthly_total = cursor.fetchone()["monthly_revenue"]

    cursor.execute("""
        SELECT total_revenue, total_orders
        FROM vw_monthly_revenue
        WHERE ym = '2025-12'
    """)
    december = cursor.fetchone()

    cursor.close()

    tolerance = Decimal("0.01")

    checks = {
        "All revenue equals dated plus undated revenue":
            abs(
                order_totals["all_revenue"]
                - order_totals["dated_revenue"]
                - order_totals["undated_revenue"]
            ) < tolerance,

        "Product revenue matches all-order revenue":
            abs(
                product_total - order_totals["all_revenue"]
            ) < tolerance,

        "Monthly revenue matches dated-order revenue":
            abs(
                monthly_total - order_totals["dated_revenue"]
            ) < tolerance,

        "December 2025 revenue matches source":
            december is not None
            and december["total_revenue"] == Decimal("12794588.15"),

        "December 2025 order count matches source":
            december is not None
            and december["total_orders"] == 268,
    }

    for name, passed in checks.items():
        print(f"{'PASS' if passed else 'FAIL'} | {name}")

finally:
    connection.close()

PASS | All revenue equals dated plus undated revenue
PASS | Product revenue matches all-order revenue
PASS | Monthly revenue matches dated-order revenue
PASS | December 2025 revenue matches source
PASS | December 2025 order count matches source


In [26]:
import time

def generate_with_retry(prompt):
    for attempt in range(2):
        try:
            return client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=prompt
            )

        except Exception as exc:
            error = str(exc)

            is_temporary = (
                "503" in error
                or "UNAVAILABLE" in error
                or "429" in error
            )

            if is_temporary and attempt == 0:
                print("Temporary API issue; retrying once...")
                time.sleep(3)
                continue

            print("AI request failed. Check API availability and limits.")
            raise RuntimeError("Gemini request unsuccessful.") from None

In [30]:
# Use the existing sample customer for this demo.
# Customer ID selection will be added later, as planned.

demo_customer_id = sample_customer_id

# Test 1: Retrieve trusted customer data from MySQL.
demo_context = fetch_customer_context(demo_customer_id)
customer = demo_context.get("vw_customer_360")

if not customer:
    raise ValueError("Customer data could not be retrieved.")

print("MySQL retrieval: PASS")

# Test 2: Generate an AI-powered Customer 360 summary.
demo_summary = generate_customer_360_summary(demo_customer_id)

if not demo_summary:
    raise ValueError("Customer summary generation returned no output.")

print("\nCUSTOMER 360 AI SUMMARY")
print(demo_summary)

# Test 3: Answer a business question using retrieved SQL context.
demo_question = "Which product category generates the most revenue?"
demo_answer = answer_business_question(demo_question)

if not demo_answer:
    raise ValueError("Business Q&A returned no output.")

print("\nBUSINESS Q&A")
print("Question:", demo_question)
print("Answer:", demo_answer)

MySQL retrieval: PASS

CUSTOMER 360 AI SUMMARY
1. Customer Snapshot
- Customer lifetime spanned 253 days, with the first order on 2024-04-05 and the last order on 2024-12-14.
- Categorized in the "3-5 Orders" purchase frequency segment, "High" engagement segment, and "No Tickets" support segment.

2. Purchase Behaviour
- Placed a total of 3 orders with a total quantity of 3.0.
- Generated a total revenue of 117,943.26, resulting in an average order value of 39,314.42.
- Reported days since the last order is 352.

3. Engagement
- Recorded 11 engagement events.

4. Support Experience
- Logged 0 support tickets, meaning positive, negative, and neutral ticket counts are all 0.0.

5. Suggested Next Action
- Consider reaching out to re-engage the customer, given the 352 days since the last order despite a history of high engagement events.

6. Data Limitations
- Average resolution time is unavailable (null) due to the absence of support tickets.

BUSINESS Q&A
Question: Which product category